In [11]:
import plotly.express as px
import xarray as xr
import numpy as np
import glob
import os
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image

In [12]:
#rname = "bsublattice_horizontal"
rname = "bsublattice_horizontal"

truep = r"/mnt/c/Users/ander/OneDrive/Documents/GitHub/QPI-Scattering/TBCalcs/"
fname = "/10x10_blattice_sep=1-rfilter1.png"

dfolder1 = r"Filtering/" + rname + "/axis1/rfilter"
dfolder2 = r"Filtering/" + rname + "/axis2/rfilter"
dfolder3 = r"Filtering/" + rname + "/axis3/rfilter"

In [13]:
narrs = []
seps_names = []

for dfolder in [dfolder1, dfolder2, dfolder3]:
    folds = [d for d in glob.glob(truep + dfolder + '/*.png')]
    basenames = [os.path.basename(fs) for fs in folds]

    # CGPT
    # Function to extract X from the filename
    def extract_number(filename):
        match = re.search(r"sep=(\d+)", filename)
        if match:
            return int(match.group(1))
        return 0  # Default value if no match found (adjust as needed)

    # Sort the filenames numerically by X
    bsort = sorted(basenames, key=extract_number)

    seps_names = [extract_number(bn) for bn in bsort]

    images = []
    for imgs in bsort:
        img = np.array(Image.open(truep + dfolder + "/" + imgs))
        images.append(img)

    narrs.append(np.array(images))

In [14]:
seps_names

[1, 3, 5, 7, 10, 12, 15]

In [16]:
fig1 = px.imshow(narrs[0], animation_frame=0, width=600, height=600)
fig2 = px.imshow(narrs[1], animation_frame=0, width=600, height=600)
fig3 = px.imshow(narrs[2], animation_frame=0, width=600, height=600)

fig1.update_layout(xaxis_showgrid=False, yaxis_showgrid=False, 
                  xaxis_visible=False, yaxis_visible=False)
fig2.update_layout(xaxis_showgrid=False, yaxis_showgrid=False, 
                  xaxis_visible=False, yaxis_visible=False)
fig3.update_layout(xaxis_showgrid=False, yaxis_showgrid=False, 
                  xaxis_visible=False, yaxis_visible=False)

fig = make_subplots(rows=1, cols=3)

# Add the first animation to the first subplot
for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)

# Add the second animation to the second subplot
for trace in fig2.data:
    fig.add_trace(trace, row=1, col=2)

for trace in fig3.data:
    fig.add_trace(trace, row=1, col=3)

frames = []

# Sync the animations by combining corresponding frames from fig1 and fig2
for i, frame in enumerate(fig1.frames):
    combined_frame = go.Frame(
        data=frame.data + fig2.frames[i].data + fig3.frames[i].data,  # Combine data from both frames
        name=str(i)
    )
    frames.append(combined_frame)

# Assign the list of frames to the figure
fig.frames = frames

# Add a slider to control the animation frames
sliders = [{
    "steps": [{
        "args": [[str(i)],  # Reference frame by name
                 {"frame": {"duration": 0, "redraw": True},
                  "mode": "immediate"}],
        "label": "Separation:" + str(seps_names[i]),
        "method": "animate"} for i in range(len(frames))],
    "active": 0,
    "transition": {"duration": 0},
    "x": 0.1,
    "xanchor": "left",
    "y": -0.1,
    "yanchor": "top",
    "len": 0.9
}]

# Update layout to include the slider
fig.update_layout(
    sliders=sliders,
    height=500, width=1000,
    xaxis_showgrid=False, yaxis_showgrid=False, 
                  xaxis_visible=False, yaxis_visible=False
)

# Show the figure
fig.update_layout(title_text="Filtered LDOS: Seperation of two b sublattice vacancy defects on graphene.")

# Show the figure
#fig.show()
fig.update_xaxes(showgrid=False, visible=False, row=1, col=1)
fig.update_yaxes(showgrid=False, visible=False, row=1, col=1)

fig.update_xaxes(showgrid=False, visible=False, row=1, col=2)
fig.update_yaxes(showgrid=False, visible=False, row=1, col=2)

fig.update_xaxes(showgrid=False, visible=False, row=1, col=3)
fig.update_yaxes(showgrid=False, visible=False, row=1, col=3)
fig.write_html(rname + "-new-plots.html")